In [3]:
"""
A specialized version to run the job on the IBM cloud.
This version submits all the jobs without waiting for results.
The jobs are saved via a tracker object for later retrieval.
"""

from qiskit import QuantumCircuit
from qiskit.compiler import transpile
from qiskit_aer import AerSimulator
from matplotlib import pyplot as plt
import numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.quantum_info import Operator

from tqdm import tqdm

In [4]:
from pmns import PMNS_matrix_4x4
from hamiltonians import U1_omega_Omega, theta_ij, J_ij
from hamiltonians import nunu_interaction_p, nunu_interaction, nunu_interaction_s
from helpers import get_target_state, swap_correction_order_bp, get_interaction_pair_index, count_2q

from backend import get_sampler
from transpile import produce_transpile_args

import sys

sys.path.append('../utils/')

from JobResult import JobResult
from DecoherenceRenormalizer import DecoherenceRenormalizer as DR
from Collector import Collector

from RunSaver import RunSaver

from RunTracker import RunTracker

In [ ]:
rs = RunSaver("4nu_t56_torino")

savepath = 'results'

run_name = '4nu_t56_torino'

SAVEFLAG = True

transpile_config_path = "config/default.json"

# backend settings
# aer, sherbrooke, fez, or ibm, or marrakesh

provider = 'ibm_torino'
sampler, backend, config = get_sampler(provider)
N = 4

# trotter settings
# set in natural units here
# conversion to mu^-1 happens later automatically
# dt = 5
T_max = 5.6
steps = 5

# run settings

shots = 8192

transpile_args = produce_transpile_args(config=config, path=transpile_config_path)
print(f"Transpile args: {transpile_args}")


rs.save_multiple([
    ('savepath', savepath),
    ('run_name', run_name),
    ('provider', provider),
    ('transpile_config_path', transpile_config_path),
    ('shots', shots),
    ('steps', steps),
    ('T_max', T_max),
])

Using IBM Torino backend
Transpile args: {'optimization_level': 3, 'basis_gates': ['cz', 'id', 'delay', 'measure', 'reset', 'rz', 'sx', 'x'], 'initial_layout': None, 'approximation_degree': 1}


In [6]:
#### Defining PMNS matrix and parameters for the Hamiltonian ####
theta_12 = np.radians(33.67)
theta_13 = np.radians(8.58)
theta_23 = np.radians(42.3)
delta_CP = np.radians(232)

U_PMNS_4x4 = PMNS_matrix_4x4(theta_12, theta_13, theta_23, delta_CP)
assert np.allclose(U_PMNS_4x4.conj().T @ U_PMNS_4x4, np.eye(4)), "PMNS matrix is not unitary"

dm21 = 7.41e-5
dm31 = 2.505e-3
E = 1e-3

delta_m21 = dm21
delta_m31 = dm31
delta_m32 = delta_m31 - delta_m21
#mu=dm31*N/(2*E) #* 1e-8
mu = dm31 * N / (2 * E)
#omega = np.sqrt(delta_m21**2 + ((delta_m31 + delta_m32)**2) / 3) / (4 * E)
omega=dm21 / (2 * E)  
Omega= dm31 / (2 * E)
#B3 = delta_m21 / np.sqrt(delta_m21**2 + ((delta_m31 + delta_m32)**2) / 3)
#B8 = (delta_m31 + delta_m32) / (np.sqrt(3) * np.sqrt(delta_m21**2 + ((delta_m31 + delta_m32)**2) / 3))
B3=0.025483
B8=0.999567
B = np.zeros(8)
B[2] = B3
B[7] = B8

T_max = T_max / mu
times = np.linspace(0, T_max, steps)

rs.save_multiple([
    ('dm21', dm21),
    ('dm31', dm31),
    ('E', E),
    ('mu', mu),
    ('omega', omega),
    ('Omega', Omega),
    ('B3', B3),
    ('B8', B8),
    ('B', B)
])

In [7]:
def apply_nunu_interaction(qc, pair_indices, order, base_pattern, dt, interact_func = nunu_interaction, post_fix_func = None):
    """
    interact_func: function for the interaction, for example, nunu_interaction or nunu_interaction_p
    post_fix_func: function to update the order and base_pattern after interaction, given by (order, base_pattern, i, j) -> (new_order, new_base_pattern)
    """
    for i, j in pair_indices:
        q = [2*i, 2*i+1, 2*j, 2*j+1]
        alpha_ij = J_ij(order[i], order[j], E, mu, N) * dt

        interact_func(qc, q, alpha_ij)
        order, base_pattern = post_fix_func(order, base_pattern, i, j) if post_fix_func else (order, base_pattern)
        
    return order, base_pattern


In [8]:
# initialization for run


def init_order_and_base_pattern():
    # generate initial pattern
    base_pattern = []
    for i in range(N):
        if i % 4 == 1:
            base_pattern.append('M')  # nu_mu
        elif i % 4 == 3:
            base_pattern.append('T')  # nu_tau
        else:
            base_pattern.append('E')  # nu_e

    # order map
    order = [i for i in range(N)]

    return order, base_pattern

# interaction sequence
method = "bubble"

pair_indices = get_interaction_pair_index(N, method = method)

rs.save("pair_indices", method)

collector = Collector(run_name)

rs.save("Type", "swap")
rs.log("logs")


In [9]:
rt = RunTracker()
run_track_save_path = "../sessions"

rs.save("run_track_save_path", run_track_save_path)

RunTracker initialized.


In [10]:
for t in times:
    collector.add("t", t * mu)

    dt = t
    print("############################################")
    print("Time step:", t)

    order, base_pattern = init_order_and_base_pattern()
    qc = QuantumCircuit(2 * N, 2 * N)

    # Initial state: alternate nu_e, nu_mu, nu_e, nu_tau pattern
    for i in range(N):
        if i % 4 == 1:
            qc.x(2*i + 1)  # nu_mu = |01⟩
        elif i % 4 == 3:
            qc.x(2*i)      # nu_tau = |10⟩

    # Apply U†_PMNS
    Udag = Operator(U_PMNS_4x4.conj().T)
    for i in range(N):
        qc.unitary(Udag, [2*i, 2*i+1], label="U†_PMNS")

    # interaction
    for i in range(N):
        qc.append(U1_omega_Omega(dt, omega, Omega), [2*i, 2*i+1])

    order, base_pattern = apply_nunu_interaction(qc, pair_indices, order, base_pattern, dt,
                                                    interact_func=nunu_interaction_p,
                                                    post_fix_func=swap_correction_order_bp)
                                                    
    # order, base_pattern = apply_nunu_interaction(qc, pair_indices, order, base_pattern, dt,
    #                                              interact_func=nunu_interaction,
    #                                              post_fix_func=None)

    # Apply U_PMNS
    # U = Operator(U_PMNS_4x4)
    for i in range(N):
        qc.unitary(Udag, [2*i, 2*i+1], label="U†_PMNS").inverse()

    # Measurement
    qc.measure(range(2 * N), reversed((range(2 * N))))

    transpile_args = produce_transpile_args(config=config, path=transpile_config_path)

    tqc = transpile(qc, backend, **transpile_args)

    print("Transpiled circuit has {0} two-qubit gates.".format(count_2q(tqc)))

    dr = DR(tqc, verbose=True)
    print("Identity circuit estimate has {0} two-qubit gates.".format(count_2q(dr.get_identity_circuit())))

    estimate_job_id = dr.estimate_error_rate_no_wait(service=sampler, transpile_options=transpile_args)

    rt.add_job("identity", estimate_job_id, "0"*(2*N))

    job = JobResult(sampler, verbose=True)

    job.run(tqc, {'shots': shots})

    job_id = job.get_job_id()

    rt.add_job("circuit", job_id, get_target_state(base_pattern))

print("All jobs submitted")

rt.dump(run_track_save_path, run_name)

if SAVEFLAG:
    collector.save(path=savepath)



############################################
Time step: 0.0
Transpiled circuit has 166 two-qubit gates.
Identity circuit estimate has 166 two-qubit gates.
Estimating error rate with 1024 shots
Job ID: d0rhr0i4p3dc73f1d77g submitted
Added job: identity_0 with id: d0rhr0i4p3dc73f1d77g and target state: 00000000
Job ID: d0rhr0vp51os73av22bg submitted
Added job: circuit_0 with id: d0rhr0vp51os73av22bg and target state: 10000100
############################################
Time step: 0.11177644710578842
Transpiled circuit has 234 two-qubit gates.
Identity circuit estimate has 234 two-qubit gates.
Estimating error rate with 1024 shots
Job ID: d0rhr17p51os73av22c0 submitted
Added job: identity_1 with id: d0rhr17p51os73av22c0 and target state: 00000000
Job ID: d0rhr16mu1ms73d4ihr0 submitted
Added job: circuit_1 with id: d0rhr16mu1ms73d4ihr0 and target state: 10000100
############################################
Time step: 0.22355289421157684
Transpiled circuit has 242 two-qubit gates.
Identity